# Notebook 3 — Classification


## Learning objectives

- Define classification and compare it with regression.
- Write down binary and multinomial logistic regression and the cross-entropy loss.
- Train a 3-class and a 6-class $\text{PM}_{2.5}$ classifier and read confusion matrices.
- Compute accuracy, per-class precision/recall/F1, macro-F1 and weighted-F1.
- Use stratified sampling for class-imbalanced data.

## 3.1 Definition

**Classification** is supervised learning whose target $y$ is categorical: $y \in \{c_{1}, c_{2}, \dots, c_{K}\}$ for a finite set of $K$ classes. When $K = 2$ we speak of *binary* classification; when $K > 2$, *multi-class*.

This notebook uses two multi-class problems built from $\text{PM}_{2.5}$:

- `target_category_3` — **Low** ($\text{PM}_{2.5}\le 35$), **Medium** ($35<\text{PM}_{2.5}\le 75$), **High** ($\text{PM}_{2.5}>75$).
- `target_category_6` — **Excellent**, **Good**, **Lightly**, **Moderately**, **Heavily**, **Severely Polluted** with thresholds $35, 75, 115, 150, 250$.

The thresholds are *not* the official Chinese AQI scheme; they are pedagogical bins chosen to give a workable class distribution.

## 3.2 Why not just use regression and threshold?

A reasonable first instinct is: regress $\text{PM}_{2.5}$, then bin the prediction to obtain a class. Why is that suboptimal?

- The squared-error loss does not align with classification metrics. A regression model that is consistently 5 μg/m³ too low scores well on RMSE, but if every borderline 36 μg/m³ becomes a predicted 31 μg/m³, every Medium hour gets misclassified as Low.
- A native classifier outputs a probability for *every* class, enabling thresholded and risk-aware decisions.
- Classifiers can use loss functions specifically designed for categorical outputs (cross-entropy).

## 3.3 The logistic regression model

### 3.3.1 Binary logistic regression

Given $\mathbf{x} \in \mathbb{R}^{d}$ and a binary label $y \in \{0,1\}$, we model the probability of class 1 as

$$P(y = 1 \mid \mathbf{x}) = \sigma\bigl(w_{0} + \mathbf{w}^{\top}\mathbf{x}\bigr), \qquad \sigma(z) = \frac{1}{1 + e^{-z}}.$$

The function $\sigma$, the **logistic** or **sigmoid** function, maps real numbers into $(0,1)$, monotonically and smoothly.

Equivalently,

$$\log \frac{P(y=1\mid\mathbf{x})}{P(y=0\mid\mathbf{x})} = w_{0} + \mathbf{w}^{\top}\mathbf{x}.$$

The log-odds is *linear* in $\mathbf{x}$ — hence "logistic *regression*". The decision boundary $P = 0.5$ corresponds to $w_{0} + \mathbf{w}^{\top}\mathbf{x} = 0$, a hyperplane in the feature space.

### 3.3.2 Multinomial (softmax) logistic regression

For $K > 2$ classes we replace the sigmoid by the **softmax** function. Each class $k$ has its own weight vector $\mathbf{w}_{k}$ and intercept $w_{0,k}$, and

$$P(y = k \mid \mathbf{x}) = \frac{\exp\bigl(w_{0,k} + \mathbf{w}_{k}^{\top}\mathbf{x}\bigr)}{\sum_{j=1}^{K} \exp\bigl(w_{0,j} + \mathbf{w}_{j}^{\top}\mathbf{x}\bigr)}.$$

Softmax produces a valid probability distribution: each entry is in $(0,1)$ and they sum to 1. The predicted class is $\arg\max_{k} P(y=k\mid\mathbf{x})$.

### 3.3.3 Loss function: cross-entropy

The natural loss for classification is the **negative log-likelihood**, also known as **cross-entropy**:

$$L(\mathbf{w}) = -\frac{1}{N} \sum_{i=1}^{N} \sum_{k=1}^{K} \mathbb{1}[y_{i}=k] \log P(y_{i}=k\mid\mathbf{x}_{i}; \mathbf{w}).$$

Cross-entropy penalises confident *wrong* predictions heavily — they push $\log P \to -\infty$. There is no closed-form minimiser; iterative solvers (L-BFGS by default in `sklearn`) optimise the loss numerically. We cap `max_iter=1000` for the harder 6-class problem.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path('data')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (9, 5)

df = pd.read_csv(DATA_DIR / 'air_quality_basic_aotizhongxin.csv', parse_dates=['datetime'])
df = df.sort_values('datetime').reset_index(drop=True)

## 3.4 Class imbalance and stratification

Air-quality categories are highly imbalanced (we saw this in Notebook 1). A naïve classifier that always predicts the majority class would still achieve a high accuracy purely from base-rate dominance.

Two countermeasures:

- **Stratified splitting**. When taking a random split, draw test rows so the class distribution is preserved.
- **Class-aware metrics**. Reporting *macro F1* alongside accuracy tells us whether the rare classes are being predicted at all (§3.6).

Because this is time-series data we use a **chronological split**, but the standard stratification idiom is:

```python
train_df, test_df = train_test_split(
    df, test_size=test_size, random_state=random_state,
    shuffle=True, stratify=df[target_column])
```

## 3.5 Worked example — pick features and split chronologically

We use the same correlation-based station selection as in Notebook 2, then build the feature matrix from $\text{PM}_{2.5}$ and $\text{PM}_{10}$ of the four chosen stations.

In [ ]:
cut = int(0.8 * len(df))
train_df, test_df = df.iloc[:cut].copy(), df.iloc[cut:].copy()

station_cols = [c for c in df.columns if c.startswith('pm25_')]
corr = train_df[['target_pm25'] + station_cols].corr()['target_pm25'].drop('target_pm25')
top4 = [c.replace('pm25_', '') for c in corr.sort_values(ascending=False).head(4).index]
feature_cols = [f'pm25_{s}' for s in top4] + [f'pm10_{s}' for s in top4]
print('Selected stations:', top4)

## 3.6 Three-class classifier

We train a multinomial logistic regression on `target_category_3`.

In [ ]:
labels3 = ['Low', 'Medium', 'High']
tr = train_df.dropna(subset=feature_cols + ['target_category_3'])
te = test_df.dropna(subset=feature_cols + ['target_category_3'])

pipe = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))])
pipe.fit(tr[feature_cols], tr['target_category_3'])
pred3 = pipe.predict(te[feature_cols])

print(f"Accuracy   : {accuracy_score(te['target_category_3'], pred3):.3f}")
print(f"Macro F1   : {f1_score(te['target_category_3'], pred3, average='macro'):.3f}")
print(f"Weighted F1: {f1_score(te['target_category_3'], pred3, average='weighted'):.3f}")
print('\nClassification report:')
print(classification_report(te['target_category_3'], pred3, labels=labels3, zero_division=0))

### Evaluating classification algorithms — the metrics

Let $\hat{y}_{i}$ be the predicted class and $y_{i}$ the true class on the test set.

**Accuracy**
$$\text{Acc} = \frac{1}{N_{\text{test}}} \sum_{i=1}^{N_{\text{test}}} \mathbb{1}[\hat{y}_{i} = y_{i}].$$
The fraction of correctly classified rows. Easy to interpret, *misleading* on imbalanced data.

**Confusion matrix**
A $K \times K$ table $C$ where $C_{k\ell}$ counts the rows whose true class is $k$ and whose predicted class is $\ell$. Diagonal entries are correct predictions; off-diagonal entries reveal which classes are systematically confused.

**Precision, recall, F1 (per class)**
For a fixed class $k$:

- $P_{k} = \dfrac{\text{TP}_{k}}{\text{TP}_{k} + \text{FP}_{k}}$ — among rows predicted $k$, what fraction were actually $k$?
- $R_{k} = \dfrac{\text{TP}_{k}}{\text{TP}_{k} + \text{FN}_{k}}$ — among rows truly $k$, what fraction did we recover?
- $F_{1,k} = 2 \dfrac{P_{k} R_{k}}{P_{k} + R_{k}}$ — the harmonic mean of precision and recall.

In atmospheric pollution alerts, the cost of false negatives (a *Heavily Polluted* hour misclassified as *Low*) usually dominates the cost of false positives. We therefore care about per-class **recall** for the dangerous classes.

**Macro and weighted F1**

$$F_{1,\text{macro}} = \frac{1}{K} \sum_{k=1}^{K} F_{1,k}, \qquad F_{1,\text{weighted}} = \frac{1}{N_{\text{test}}} \sum_{k=1}^{K} n_{k} F_{1,k}.$$

Macro-$F_{1}$ treats all classes as equally important — the right metric when we care about rare classes. Weighted-$F_{1}$ is dominated by the majority class.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cm = confusion_matrix(te['target_category_3'], pred3, labels=labels3)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels3, yticklabels=labels3, ax=axes[0])
axes[0].set_title('Confusion matrix (counts)'); axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

cm_norm = confusion_matrix(te['target_category_3'], pred3, labels=labels3, normalize='true')
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=labels3, yticklabels=labels3, ax=axes[1])
axes[1].set_title('Confusion matrix (row-normalised)'); axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')
plt.tight_layout(); plt.show()

**Reading the confusion matrix.** The `normalize='true'` option divides each row by its sum, so $C_{k\ell}$ becomes the conditional probability *given true class $k$, what fraction was predicted as $\ell$?*. This is the right normalisation when classes are imbalanced. The diagonal cells should approach 1 for a strong classifier; off-diagonal mass shows which classes are confused.

## 3.7 Six-class classifier — granularity-induced error

Same data, finer labels. The 6-class problem inserts five thresholds ($35, 75, 115, 150, 250$) where the 3-class problem inserted two ($35, 75$). Each new threshold produces a band of borderline observations whose true category is uncertain. Macro-F1 should fall.

In [ ]:
labels6 = ['Excellent', 'Good', 'Lightly Polluted', 'Moderately Polluted', 'Heavily Polluted', 'Severely Polluted']
tr = train_df.dropna(subset=feature_cols + ['target_category_6'])
te = test_df.dropna(subset=feature_cols + ['target_category_6'])

pipe6 = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))])
pipe6.fit(tr[feature_cols], tr['target_category_6'])
pred6 = pipe6.predict(te[feature_cols])

print(f"Accuracy   : {accuracy_score(te['target_category_6'], pred6):.3f}")
print(f"Macro F1   : {f1_score(te['target_category_6'], pred6, average='macro'):.3f}")
print(f"Weighted F1: {f1_score(te['target_category_6'], pred6, average='weighted'):.3f}")

In [ ]:
cm6 = confusion_matrix(te['target_category_6'], pred6, labels=labels6, normalize='true')
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm6, annot=True, fmt='.2f', cmap='Blues', xticklabels=labels6, yticklabels=labels6, ax=ax)
ax.set_title('6-class confusion matrix (row-normalised)')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.tight_layout(); plt.show()

Notice that the densest off-diagonal mass sits on the *adjacent* classes — the classifier rarely confuses *Excellent* with *Severely Polluted*, but it routinely confuses *Lightly* with *Moderately*. This is **granularity-induced error**: the harder you make the classification problem, the worse the metric, even when the underlying $\text{PM}_{2.5}$ regression has not changed.

## 3.8 Probability outputs and alert generation

Classifiers produced by `predict_proba` return per-class probabilities, which can drive thresholded alerts beyond the default argmax decision rule.

In [ ]:
proba = pipe.predict_proba(te[feature_cols])
proba_df = pd.DataFrame(proba, columns=pipe.classes_, index=te.index).head()
print('First five test rows, predicted probabilities:')
proba_df

Each row sums to 1; the predicted class is the column with the highest probability. A municipal dashboard could ingest these probabilities and emit:

- a **green** status when $P(\text{Low}) > 0.85$;
- an **amber** status when $P(\text{Medium}) > 0.50$;
- a **red** status when $P(\text{High}) > 0.30$.

The probability outputs of logistic regression are well-calibrated in the moderate-probability range; in the tail they tend to be over-confident, so post-hoc *probability calibration* (Platt scaling or isotonic regression) is sometimes added before deployment.

## 3.9 Real-world application — pollution alerts

The thresholds in `target_category_6` align loosely with the World Health Organization air-quality guidelines (24-hour mean) and the Chinese AQI scheme. The natural deployment of a classifier of this kind is **early warning** — the `predict_proba` outputs of an hour-ahead classifier (Notebook 5 covers AQI forecasting) drive automated alerts to schools, hospitals and outdoor-event organisers, with the alert threshold tuned for the cost of false negatives in that specific application.

## 3.10 Common misconceptions

- **"Higher accuracy is always better."** With a 90 / 5 / 5 % class split, predicting "majority class always" achieves 90 % accuracy and 0 % recall on the rare classes. Macro-$F_{1}$ would be approximately $1/3$, exposing the failure.
- **"The decision boundary of logistic regression is curved."** It is *linear* in feature space. The probability *output* is non-linear in $\mathbf{x}$ (the sigmoid is a curve) but the boundary $P = 0.5$ is a hyperplane. To get a curved boundary you must add non-linear features or switch to a non-linear model (Notebook 6).
- **"Cross-entropy and squared error are interchangeable losses."** They are not. Cross-entropy goes to $\infty$ when the predicted probability of the true class approaches 0; squared error is bounded.
- **"More classes = better resolution."** Only if there is real signal at that resolution. The 6-class problem doubles the granularity but the underlying 4-feature linear classifier cannot resolve the fine boundaries. Reduce the class count if macro-$F_{1}$ stagnates.

---
## Exercises

### Exercise 3.1 — alert-style threshold

*Hint: define an alert whenever `predict_proba` for the High class exceeds 0.3. How many alerts per month? What fraction were actually High? (That fraction is the alert *precision*.)*

In [ ]:
# EXERCISE 3.1
# high_idx = pipe.classes_.tolist().index('High')
# p_high = pipe.predict_proba(te[feature_cols])[:, high_idx]
# TODO: count alerts per month and compute precision (fraction of alerts that were truly High)


### Exercise 3.2 — class weights

*Hint: `LogisticRegression(class_weight='balanced')` upweights rare classes in the cross-entropy loss. Does macro-F1 improve? Does accuracy fall? (Both effects are usually present — that's the point of the trade-off.)*

In [ ]:
# EXERCISE 3.2
# pipe_bal = Pipeline([('scaler', StandardScaler()),
#                      ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))])
# TODO: fit, predict, compare metrics with the unweighted pipeline.


### Exercise 3.3 — drop *High* and re-fit

*Hint: filter rows where target_category_3 != 'High' and retrain. The 2-class problem should be easier — verify by looking at macro-F1.*

In [ ]:
# EXERCISE 3.3
# TODO: filter, refit, report metrics for the 2-class problem.


## 3.11 Chapter summary

- Classification predicts a discrete class; logistic regression generalises linear regression by passing a linear score through the sigmoid (binary) or softmax (multi-class) function.
- The native loss is cross-entropy; there is no closed-form solution, so iterative solvers fit the parameters.
- Accuracy alone is insufficient on imbalanced atmospheric data; pair it with the confusion matrix, per-class precision/recall, and macro-$F_{1}$.
- Stratified sampling preserves class balance in random splits; for time series prefer a chronological split.
- Granularity matters: the same input data yields lower metrics under 6-class than under 3-class because borderline observations multiply with each new threshold.

**Next:** Notebook 4 deepens the feature side — cyclic encoding, lag features, and the chronological split formalised.